In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import (classification_report, accuracy_score, f1_score,
                             cohen_kappa_score, matthews_corrcoef, confusion_matrix)
from keras.models import Model
from keras.layers import (LSTM, Dense, Dropout, MaxPooling1D, Input, Bidirectional,
                          Conv1D, Concatenate, Permute, Reshape, Multiply,
                          GlobalMaxPooling1D, Attention, Activation, BatchNormalization)
from keras.optimizers import RMSprop, Adam, Nadam
from keras.losses import categorical_crossentropy, mean_squared_error
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping


In [3]:
class Config:

    #Globals
     #Globals
    batch_size = 16
    num_classes = 2  # classes, seizure/no seizure
    epochs = 25   # Epoch iterations
    time_step_length = 5
    row_hidden = 128  # hidden neurons in conv layers
    col_hidden = 128   # hidden neurons in the Bi-LSTM layers
    RANDOM_SEED = 3333    
    N_TIME_STEPS = 125   # 50 records in each sequence
    N_FEATURES = 3     # mag,hr,roi_Ratio,output
    step = 100           # window overlap = 50 -10 = 40  (80% overlap)
    N_CLASSES = 2     # class label
    learning_rate = 0.0000001
    k = 5 # number of k folds
    target_class_count=58250

In [4]:
import pandas as pd
import numpy as np

class DataFrameWithFFT:
    def __init__(self, csv_path, time_steps, sampling_rate=1.0):
        """
        Initialize the class with the path to the CSV data and additional parameters.
        
        :param csv_path: Path to the CSV file containing raw data.
        :param time_steps: Number of time steps for each FFT calculation.
        :param sampling_rate: Sampling rate for calculating the FFT.
        """
        self.csv_path = csv_path
        self.time_steps = time_steps
        self.sampling_rate = sampling_rate
        self.dataframe = None

    def calculate_fft(self, raw_data_window):
        """
        Calculate the FFT for a given window of raw data.
        
        :param raw_data_window: Array of raw data values for the time window.
        :return: FFT magnitude values (only positive frequencies).
        """
        raw_data_window = np.array(raw_data_window) - np.mean(raw_data_window)  # Remove DC component
        fft_result = np.fft.fft(raw_data_window)  # Compute FFT
        fft_magnitude = np.abs(fft_result)  # Compute magnitude
        positive_fft_magnitude = fft_magnitude[:len(fft_magnitude) // 2]  # Take positive half
        return positive_fft_magnitude

    def create_dataframe_with_fft(self):
        """
        Create a dataframe where each row corresponds to a single FFT value
        for a given time window of rawData.
        
        :return: Expanded dataframe with FFT values.
        """
        # Load CSV data
        df = pd.read_csv(self.csv_path)

        # Ensure rawData exists in the dataframe
        if 'rawData' not in df.columns:
            raise ValueError("The CSV file must contain a 'rawData' column.")

        # Sliding window FFT calculation
        expanded_rows = []
        for i in range(0, len(df) - self.time_steps + 1, self.time_steps):
            raw_data_window = df['rawData'].values[i:i + self.time_steps]  # Extract a time window
            fft_values = self.calculate_fft(raw_data_window)

            # Create rows for each FFT value
            for idx, fft_value in enumerate(fft_values):
                row = df.iloc[i].copy()  # Copy metadata from the first row of the window
                row['FFT'] = fft_value  # Assign the current FFT value
                row['FFT_Index'] = idx  # Add an index for FFT value
                expanded_rows.append(row)

        # Create new dataframe from expanded rows
        expanded_df = pd.DataFrame(expanded_rows)

        # Drop unnecessary columns for simplicity
        self.dataframe = expanded_df.reset_index(drop=True)
        return expanded_df


In [5]:
csv_path = '../Data/ipd_osdb_dataset.csv'
time_steps = 10  # Define the number of time steps for each FFT calculation
data_processor = DataFrameWithFFT(csv_path, time_steps, sampling_rate=125)

# Create the dataframe with FFT column
processed_df = data_processor.create_dataframe_with_fft()
processed_df.head()



,Id,eventId,userID,rawData,ppg,type,subType,label,FFT,FFT_Index
0,1,5635,45,1630.0,69.0,Seizure,Aura,0,0.000000,0
0,1,5635,45,1630.0,69.0,Seizure,Aura,0,20.088985,1
0,1,5635,45,1630.0,69.0,Seizure,Aura,0,16.590637,2
0,1,5635,45,1630.0,69.0,Seizure,Aura,0,13.580599,3
0,1,5635,45,1630.0,69.0,Seizure,Aura,0,13.955314,4


In [6]:
class DataLoader:
    def __init__(self, dataframe, time_steps, step, target_column):
        self.dataframe = dataframe
        self.time_steps = time_steps
        self.step = step
        self.target_column = target_column

    def load_data(self):
        segments = []
        labels = []
        event_ids = []
        user_ids = []

        # Group data by eventID to ensure events are kept intact
        grouped = self.dataframe.groupby('eventId')

        for event_id, group in grouped:
            if len(group) >= self.time_steps:  # Process if the event group has enough data
                for i in range(0, len(group) - self.time_steps + 1, self.step):
                    mag = group['rawData'].values[i: i + self.time_steps]
                    hr = group['ppg'].values[i: i + self.time_steps]
                    FFT = group['FFT'].values[i: i + self.time_steps]
                    segment = np.column_stack((mag, hr, FFT))  # Combine magnitude and heart rate features
                    label_mode = stats.mode(group[self.target_column][i: i + self.time_steps])
                    if isinstance(label_mode.mode, np.ndarray):
                        label = label_mode.mode[0]
                    else:
                        label = label_mode.mode

                    segments.append(segment)
                    labels.append(label)
                    event_ids.append(event_id)
                    user_ids.append(group['userID'].iloc[0])  # Assuming userID is consistent within an event

        # Convert to numpy arrays
        segments = np.asarray(segments, dtype=np.float32)
        labels = np.asarray(pd.get_dummies(labels), dtype=np.float32)

        # Create DataFrame to store eventID and userID alongside segments and labels
        df_labels = pd.DataFrame({
            'segments': list(segments),
            'labels': list(labels),
            'eventId': event_ids,
            'userID': user_ids
        })

        return df_labels
    
    
import numpy as np
import pandas as pd
from random import sample

class DataFormatter:
    def __init__(self, config):
        self.config = config

    def format_data(self, df_labels, test_split=0.3):
        """
        Split data into training and testing sets based on eventID, while ensuring proper row order is maintained.

        :param df_labels: DataFrame containing the dataset with 'eventId' column.
        :param test_split: Fraction of data to use for testing.
        :return: X_train_reshaped, X_test_reshaped, y_train, y_test
        """
        # Strip any unwanted spaces in column names (to ensure there are no hidden characters)
        df_labels.columns = df_labels.columns.str.strip()

        # Check if 'eventId' column is present
        if 'eventId' not in df_labels.columns:
            raise ValueError("The DataFrame must contain an 'eventId' column.")

        # Validate segment structure
        if not all(isinstance(seg, list) and len(seg) == 125 for seg in df_labels['segments']):
            raise ValueError("Each segment in the 'segments' column must be a list of 125 elements.")

        # Group by 'eventId' to maintain event-wise split
        unique_event_ids = df_labels['eventId'].unique()
        num_test_event_ids = int(len(unique_event_ids) * test_split)
        test_event_ids = sample(list(unique_event_ids), num_test_event_ids)
        test_event_ids_set = set(test_event_ids)

        # Split into training and testing sets
        df_test = df_labels[df_labels['eventId'].isin(test_event_ids_set)]
        df_train = df_labels[~df_labels['eventId'].isin(test_event_ids_set)]

        # Extract segments and labels for training and testing sets
        X_train = np.asarray(df_train['segments'].tolist(), dtype=np.float32)
        y_train = np.asarray(df_train['labels'].tolist(), dtype=np.float32)
        X_test = np.asarray(df_test['segments'].tolist(), dtype=np.float32)
        y_test = np.asarray(df_test['labels'].tolist(), dtype=np.float32)

        # Reshape the segments to include feature dimension
        X_train_reshaped = self._reshape_segments(X_train)
        X_test_reshaped = self._reshape_segments(X_test)

        return X_train_reshaped, X_test_reshaped, y_train, y_test

    def _reshape_segments(self, segments):
        """
        Reshapes the segments array into the required format with feature dimension.
        :param segments: The array of segments to reshape.
        :return: Reshaped segments in a dictionary format with feature keys.
        """
        reshaped_segments = {}
        num_samples, num_time_steps, num_features = segments.shape
        for i in range(num_features):
            feature_name = f"Feature_{i+1}"
            reshaped_segments[feature_name] = segments[:, :, i].reshape(-1, num_time_steps, 1)
        return reshaped_segments


In [34]:
import numpy as np
import pandas as pd
from random import sample

class DataFormatter:
    def __init__(self, config):
        self.config = config

    def format_data(self, df_labels, test_split=0.3):
        """
        Split data into training and testing sets based on eventID, while ensuring proper row order is maintained.

        :param df_labels: DataFrame containing the dataset with 'eventId' column.
        :param test_split: Fraction of data to use for testing.
        :return: X_train_reshaped, X_test_reshaped, y_train, y_test
        """
        # Strip any unwanted spaces in column names (to ensure there are no hidden characters)
        df_labels.columns = df_labels.columns.str.strip()

        # Check if 'eventId' column is present
        if 'eventId' not in df_labels.columns:
            raise ValueError("The DataFrame must contain an 'eventId' column.")

        # Validate segment structure
        if not all(isinstance(seg, np.ndarray) and seg.shape == (125, 3) for seg in df_labels['segments']):
            raise ValueError("Each segment in the 'segments' column must be an array of shape (125, 3).")

        # Group by 'eventId' to maintain event-wise split
        unique_event_ids = df_labels['eventId'].unique()
        num_test_event_ids = int(len(unique_event_ids) * test_split)
        test_event_ids = sample(list(unique_event_ids), num_test_event_ids)
        test_event_ids_set = set(test_event_ids)

        # Split into training and testing sets
        df_test = df_labels[df_labels['eventId'].isin(test_event_ids_set)]
        df_train = df_labels[~df_labels['eventId'].isin(test_event_ids_set)]

        # Extract segments and labels for training and testing sets
        X_train = np.stack(df_train['segments'].values)
        y_train = np.stack(df_train['labels'].values)
        X_test = np.stack(df_test['segments'].values)
        y_test = np.stack(df_test['labels'].values)

        return X_train, X_test, y_train, y_test

    def _reshape_segments(self, segments):
        """
        Reshapes the segments array into the required format with feature dimension.
        :param segments: The array of segments to reshape.
        :return: Reshaped segments in a dictionary format with feature keys.
        """
        reshaped_segments = {}
        num_samples, num_time_steps, num_features = segments.shape
        for i in range(num_features):
            feature_name = f"Feature_{i+1}"
            reshaped_segments[feature_name] = segments[:, :, i].reshape(-1, num_time_steps, 1)
        return reshaped_segments


In [35]:
import os
import tensorflow as tf

# Set the environment variable to suppress info logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Now initialize TensorFlow (this will suppress info logs)
tf.get_logger().setLevel('ERROR')

from tensorflow.keras.layers import (
    Layer, Add, Input, Conv1D, BatchNormalization, Activation,
    MaxPooling1D, Bidirectional, LSTM, Dense, Dropout,
    Reshape, Permute, Attention, GlobalMaxPooling1D, Concatenate, MultiHeadAttention
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras.regularizers import l2
import tensorflow as tf
import json
from tensorflow.keras.models import save_model as tf_save_model, load_model as tf_load_model
from tensorflow.keras.saving import register_keras_serializable

# Register the custom layer to make it serializable
@register_keras_serializable(package='custom', name='EnhancedFusionLayer')
class EnhancedFusionLayer(Layer):
    def __init__(self, num_heads, key_dim, **kwargs):
        super(EnhancedFusionLayer, self).__init__(**kwargs)
        self.num_heads = num_heads  # Store num_heads as an attribute
        self.key_dim = key_dim      # Store key_dim as an attribute
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
        
    def call(self, inputs):
        # Concatenate inputs along the last axis
        concatenated_inputs = Concatenate()(inputs)
        # Apply multi-head attention to concatenated inputs
        attention_output = self.attention(concatenated_inputs, concatenated_inputs)
        # Add the original concatenated inputs to the attention output
        return Add()([concatenated_inputs, attention_output])
        
    def get_config(self):
        # Retrieve base config and update with num_heads and key_dim
        config = super(EnhancedFusionLayer, self).get_config()
        config.update({
            "num_heads": self.num_heads,  # Use stored attribute
            "key_dim": self.key_dim       # Use stored attribute
         })
        return config


In [9]:
import json
import os
import tensorflow as tf

# Set the environment variable to suppress info logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Now initialize TensorFlow (this will suppress info logs)
tf.get_logger().setLevel('ERROR')

import tensorflow as tf
from keras.layers import LSTM, Dense, Dropout, MaxPooling1D, Input, Bidirectional, Conv1D, Concatenate, Permute, Reshape, Multiply, GlobalMaxPooling1D, Attention, Activation, BatchNormalization
from keras.optimizers import RMSprop, Adam
from keras.losses import categorical_crossentropy
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.saving import register_keras_serializable
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.models import save_model as tf_save_model, load_model as tf_load_model
from tensorflow.keras.utils import plot_model

class Amber_RF:
    def __init__(self, row_hidden, col_hidden, num_classes):
        self.row_hidden = row_hidden
        self.col_hidden = col_hidden
        self.num_classes = num_classes
        self.model = None

    def conv_block(self, in_layer, filters, kernel_size):
        conv = Conv1D(filters=filters, kernel_size=kernel_size, padding='same')(in_layer)
        conv = BatchNormalization()(conv)
        conv = Activation('relu')(conv)
        return conv

    def lstm_pipe(self, in_layer):
        b1 = self.conv_block(in_layer, filters=64, kernel_size=3)
        b1 = MaxPooling1D(pool_size=2)(b1)
        b2 = self.conv_block(b1, filters=64, kernel_size=3)
        b2 = MaxPooling1D(pool_size=2)(b2)
        b3 = self.conv_block(b2, filters=128, kernel_size=3)
        b3 = MaxPooling1D(pool_size=2)(b3)
        encoded_rows = Bidirectional(LSTM(self.row_hidden, return_sequences=True))(b3)
        return LSTM(self.col_hidden)(encoded_rows)

    def build_model(self, num_features, input_shape, num_heads=4, key_dim=64):
        input_layers = []
        lstm_outputs = []

        for i in range(num_features):
            input_layer = Input(shape=input_shape, name=f'input_feature_{i+1}')
            input_layers.append(input_layer)
            lstm_output = self.lstm_pipe(Permute(dims=(1, 2))(input_layer))
            lstm_output_reshaped = Reshape((-1,))(lstm_output)
            lstm_outputs.append(lstm_output_reshaped)

        attention_outputs = []
        for i, lstm_output in enumerate(lstm_outputs):
            lstm_output_reshaped = Reshape((-1, lstm_output.shape[-1]))(lstm_output)
            attention_output = Attention()([lstm_output_reshaped, lstm_output_reshaped])
            attention_outputs.append(attention_output)

        fused_features = EnhancedFusionLayer(num_heads=num_heads, key_dim=key_dim)(attention_outputs)

        dense_output = Dense(128, activation='relu', kernel_regularizer=l2(0.0001))(fused_features)
        dense_output = BatchNormalization()(dense_output)
        dense_output = Dropout(0.1)(dense_output)
        prediction = Dense(self.num_classes, activation='softmax')(GlobalMaxPooling1D()(dense_output))

        self.model = Model(inputs=input_layers, outputs=prediction)

    def compile_model(self):
        optimizer = Adam(learning_rate=Config.learning_rate, epsilon=1e-9)
        #self.model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
        self.model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['accuracy'])


    def train_model(self, X_train_list, y_train, X_val_list, y_val):
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5, verbose=1)
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)

        history = self.model.fit(
            X_train_list, y_train,
            validation_data=(X_val_list, y_val),
            epochs=Config.epochs ,
            batch_size=Config.batch_size,
            verbose=1,
            callbacks=[reduce_lr, early_stopping]
        )
        return history

    def save_model(self, path):
        """Save the model, including custom layers, to a .keras file."""
        # Save the model architecture and weights
        self.model.save(path)
        
        # Extract custom layer metadata from the existing layers in the model
        custom_objects_metadata = {}
        for layer in self.model.layers:
            if isinstance(layer, EnhancedFusionLayer):
                custom_objects_metadata[layer.name] = layer.get_config()

        # Save custom layers metadata (if any)
        with open(f"{path}_custom_objects.json", "w") as file:
            json.dump(custom_objects_metadata, file)
            

    @staticmethod
    def load_model(path):
        # Load custom layer metadata
        with open(f"{path}_custom_objects.json", "r") as file:
            custom_objects_metadata = json.load(file)
        
        # Load the model architecture and weights, specifying the custom layers
        model = tf_load_model(path, custom_objects={**custom_objects_metadata})
        
        # If needed, instantiate Amber_RF with proper parameters
        # Assuming the model's class properties are fixed, e.g.:
        amber_model = Amber_RF(row_hidden=Config.row_hidden, col_hidden=Config.col_hidden, num_classes=2)
        amber_model.model = model  # Assign the loaded model to amber_model
        
        return amber_model


    def evaluate_model(self, X_test, y_test, batch_size=Config.batch_size):
        return self.model.evaluate(X_test, y_test, batch_size=Config.batch_size)

    def predict(self, X):
        return self.model.predict(X)

    def architecture(self):
        return self.model.summary()
    
    def visualize_model(self, save_path='model_architecture.png'):
        """
        Save a visual representation of the model architecture to a PNG file.

        Args:
            save_path (str): The file path to save the PNG image.
        """
        if self.model is None:
            print("Model has not been built yet. Please build the model first.")
        else:
            try:
                plot_model(self.model, to_file=save_path, show_shapes=True, show_layer_names=True)
                print(f"Model architecture saved as {save_path}.")
            except ImportError as e:
                print("Could not generate model plot. Ensure Graphviz is installed. Error:", e)


In [10]:
class KFoldCrossValidation:
    def __init__(self, ts_model, X_train, y_train, batch_size=Config.batch_size, epochs=Config.epochs, k=5, save_dir='plots'):
        self.ts_model = ts_model
        self.X_train = X_train
        self.y_train = y_train
        self.batch_size = batch_size
        self.epochs = epochs
        self.k = k
        self.save_dir = save_dir
        os.makedirs(self.save_dir, exist_ok=True)  # Create directory if it doesn't exist
        self.history_accumulated = {"accuracy": [], "loss": [], "val_accuracy": [], "val_loss": []}  # Initialize empty dictionaries to accumulate metrics
        self.fold_history = []  # Initialize list to store individual fold histories

    def plot_confusion_matrix(self, fold, confusion_mat):
        plt.figure(figsize=(8, 6))
        sns.heatmap(confusion_mat, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'Confusion Matrix - Fold {fold + 1}')
        plt.xlabel('Predicted Labels')
        plt.ylabel('True Labels')
        plt.savefig(os.path.join(self.save_dir, f'confusion_matrix_fold_{fold + 1}.png'))  # Save each plot with a unique filename
        plt.close()

    def plot_individual_metrics(self, fold):
        fig, axs = plt.subplots(1, 2, figsize=(12, 6))  # Create subplots for accuracy and loss

        # Plot training accuracy
        axs[0].plot(self.fold_history[fold]['accuracy'], label='Training Accuracy')
        axs[0].plot(self.fold_history[fold]['val_accuracy'], label='Validation Accuracy')
        axs[0].set_title('Accuracy - Fold {}'.format(fold + 1))
        axs[0].set_xlabel('Epoch')
        axs[0].set_ylabel('Accuracy')
        axs[0].legend()

        # Plot training loss
        axs[1].plot(self.fold_history[fold]['loss'], label='Training Loss')
        axs[1].plot(self.fold_history[fold]['val_loss'], label='Validation Loss')
        axs[1].set_title('Loss - Fold {}'.format(fold + 1))
        axs[1].set_xlabel('Epoch')
        axs[1].set_ylabel('Loss')
        axs[1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, f'individual_metrics_fold_{fold + 1}.png'))  # Save the plot with a unique filename
        plt.close()  # Close the plot to avoid displaying it

    def plot_overall_metrics(self):
        fig, axs = plt.subplots(2, 2, figsize=(12, 10))  # Create subplots for accuracy, validation accuracy, loss, and validation loss

        # Plot overall accuracy
        for fold in range(self.k):
            axs[0, 0].plot(range(1, self.epochs + 1), self.fold_history[fold]['accuracy'], label=f'Fold {fold + 1}')
            axs[0, 0].set_title('Accuracy')
            axs[0, 0].set_xlabel('Epoch')
            axs[0, 0].set_ylabel('Accuracy')
            axs[0, 0].legend()

        # Plot overall validation accuracy
        for fold in range(self.k):
            axs[0, 1].plot(range(1, self.epochs + 1), self.fold_history[fold]['val_accuracy'], label=f'Fold {fold + 1}')
            axs[0, 1].set_title('Validation Accuracy')
            axs[0, 1].set_xlabel('Epoch')
            axs[0, 1].set_ylabel('Accuracy')
            axs[0, 1].legend()

        # Plot overall loss
        for fold in range(self.k):
            axs[1, 0].plot(range(1, self.epochs + 1), self.fold_history[fold]['loss'], label=f'Fold {fold + 1}')
            axs[1, 0].set_title('Loss')
            axs[1, 0].set_xlabel('Epoch')
            axs[1, 0].set_ylabel('Loss')
            axs[1, 0].legend()

        # Plot overall validation loss
        for fold in range(self.k):
            axs[1, 1].plot(range(1, self.epochs + 1), self.fold_history[fold]['val_loss'], label=f'Fold {fold + 1}')
            axs[1, 1].set_title('Validation Loss')
            axs[1, 1].set_xlabel('Epoch')
            axs[1, 1].set_ylabel('Loss')
            axs[1, 1].legend()

        plt.tight_layout()
        plt.savefig(os.path.join(self.save_dir, 'overall_metrics.png'))
        plt.close()  # Close the plot to avoid displaying it

    def run(self):
        kf = KFold(n_splits=self.k, shuffle=True)
        all_test_losses = []
        all_test_accuracies = []
        for fold, (train_index, test_index) in enumerate(kf.split(self.X_train[0])):
            print(f"Fold {fold + 1}/{self.k}")
            X_fold_train = [X[train_index] for X in self.X_train]
            y_fold_train = self.y_train[train_index]
            X_fold_val = [X[test_index] for X in self.X_train]
            y_fold_val = self.y_train[test_index]
            self.ts_model.build_model(num_features=2, input_shape=(Config.N_TIME_STEPS, 1))
            self.ts_model.compile_model()
            history = self.ts_model.train_model(X_fold_train, y_fold_train, X_fold_val, y_fold_val, epochs=Config.epochs, batch_size=Config.batch_size)
            test_loss, test_accuracy = self.ts_model.evaluate_model(X_fold_val, y_fold_val)
            all_test_losses.append(test_loss)
            all_test_accuracies.append(test_accuracy)
            print(f"Fold {fold + 1} - Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

            # Accumulate history
            self.fold_history.append(history.history)

            # Generate confusion matrix
            y_pred_val = self.ts_model.predict(X_fold_val)
            y_pred_classes = np.argmax(y_pred_val, axis=1)
            y_true_classes = np.argmax(y_fold_val, axis=1)

            confusion_mat = confusion_matrix(y_true_classes, y_pred_classes)

            # Print confusion matrix
            print(f"Confusion Matrix for Fold {fold + 1}:\n{confusion_mat}\n")

            # Save confusion matrix plot
            self.plot_confusion_matrix(fold, confusion_mat)

            # Save individual plots
            self.plot_individual_metrics(fold)

        avg_test_loss = np.mean(all_test_losses)
        avg_test_accuracy = np.mean(all_test_accuracies)
        print(f"Average Test Loss: {avg_test_loss:.4f}, Average Test Accuracy: {avg_test_accuracy:.4f}")

        # Plot overall metrics
        #self.plot_overall_metrics()

        return self.history_accumulated

In [11]:
import matplotlib.pyplot as plt
import os

class PlotKFoldMetrics:
    def __init__(self, fold_history, output_dir='kfold_results'):
        self.fold_history = fold_history
        self.output_dir = output_dir

        # Create output directory if it doesn't exist
        os.makedirs(self.output_dir, exist_ok=True)

    def plot_metrics(self):
        # Number of folds
        n_folds = len(self.fold_history)

        # Create a figure for accuracy and loss
        fig, axs = plt.subplots(2, 2, figsize=(12, 10))

        # Plot accuracy for each fold
        for fold in range(n_folds):
            axs[0, 0].plot(self.fold_history[fold]['accuracy'], label=f'Fold {fold + 1}')
            axs[0, 1].plot(self.fold_history[fold]['val_accuracy'], label=f'Fold {fold + 1}')
            axs[1, 0].plot(self.fold_history[fold]['loss'], label=f'Fold {fold + 1}')
            axs[1, 1].plot(self.fold_history[fold]['val_loss'], label=f'Fold {fold + 1}')

        # Set titles and labels for the plots
        axs[0, 0].set_title('Training Accuracy')
        axs[0, 0].set_xlabel('Epoch')
        axs[0, 0].set_ylabel('Accuracy')
        axs[0, 0].legend()

        axs[0, 1].set_title('Validation Accuracy')
        axs[0, 1].set_xlabel('Epoch')
        axs[0, 1].set_ylabel('Accuracy')
        axs[0, 1].legend()

        axs[1, 0].set_title('Training Loss')
        axs[1, 0].set_xlabel('Epoch')
        axs[1, 0].set_ylabel('Loss')
        axs[1, 0].legend()

        axs[1, 1].set_title('Validation Loss')
        axs[1, 1].set_xlabel('Epoch')
        axs[1, 1].set_ylabel('Loss')
        axs[1, 1].legend()

        # Adjust layout and save the figure
        plt.tight_layout()
        plt.savefig(os.path.join(self.output_dir, 'kfold_metrics.png'))
        plt.show()

In [12]:
class EventMetricsEvaluator:
    def __init__(self, model, events_folder, n_time_steps=125):
        self.model = model
        self.events_folder = events_folder
        self.n_time_steps = n_time_steps
        self.metrics_summary = pd.DataFrame(columns=['CSV_File', 'EventID', 'UserID', 'Accuracy', 'Sensitivity', 'False_Positive_Rate', 'False_Negative_Rate', 'False_Alarm_Rate'])
        self.user_event_counts = {}
        self.user_metrics = {}
        self.event_metrics = []

    def calculate_metrics(self, y_true, y_pred):
        TP, TN, FP, FN = 0, 0, 0, 0

        for true_label, pred_label in zip(y_true, y_pred):
            if true_label in [1, 2]:
                if pred_label in [1, 2]:
                    TP += 1
                else:
                    FN += 1
            elif true_label == 0:
                if pred_label in [1, 2]:
                    FP += 1
                else:
                    TN += 1

        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0
        false_positive_rate = FP / (FP + TN) if (FP + TN) > 0 else 0
        false_negative_rate = FN / (TP + FN) if (TP + FN) > 0 else 0
        false_alarm_rate = FP / len(y_true) if len(y_true) > 0 else 0

        return sensitivity, false_positive_rate, false_negative_rate, false_alarm_rate

    def evaluate(self):
        csv_files = [f for f in os.listdir(self.events_folder) if f.endswith('.csv')]

        for csv_file in csv_files:
            mypath = os.path.join(self.events_folder, csv_file)
            df = pd.read_csv(mypath).fillna(-1)

            segments_acceleration = []
            segments_heart_rate = []
            labels = []
            eventIDs = []
            userIDs = []
            false_alarms = 0

            for i in range(0, len(df), self.n_time_steps):
                group = df.iloc[i:i + self.n_time_steps]
                if len(group) == self.n_time_steps:
                    segment_acceleration = group['rawData'].values
                    segment_heart_rate = group['hr'].values

                    label = group['label'].mode()[0]
                    eventIDs.append(group['eventID'].values[0])
                    userIDs.append(group['userID'].values[0])

                    segments_acceleration.append(segment_acceleration)
                    segments_heart_rate.append(segment_heart_rate)
                    labels.append(label)

            # Convert segments to numpy arrays
            segments_acceleration = np.array(segments_acceleration, dtype=np.float32).reshape(-1, self.n_time_steps, 1)
            segments_heart_rate = np.array(segments_heart_rate, dtype=np.float32).reshape(-1, self.n_time_steps, 1)

            # Model predictions
            preds = self.model.predict([segments_acceleration, segments_heart_rate])
            preds_classes = np.argmax(preds, axis=-1) if preds.ndim > 1 else preds

            # Ground truth labels
            ground_truth_labels = np.array(labels)

            # False alarms calculation
            for true_label, pred_label in zip(ground_truth_labels, preds_classes):
                if true_label in [0, 2] and pred_label == 1:
                    false_alarms += 1

            total = len(preds_classes)
            overall_false_alarm_rate = false_alarms / total if total > 0 else 0

            # Calculate metrics
            sensitivity, false_positive_rate, false_negative_rate, _ = self.calculate_metrics(ground_truth_labels, preds_classes)

            # Append to metrics summary for each event
            self.metrics_summary = self.metrics_summary.append({
                'CSV_File': csv_file,
                'EventID': eventIDs[0] if eventIDs else 'N/A',
                'UserID': userIDs[0] if userIDs else 'N/A',
                'Accuracy': None,
                'Sensitivity': sensitivity,
                'False_Positive_Rate': false_positive_rate,
                'False_Negative_Rate': false_negative_rate,
                'False_Alarm_Rate': overall_false_alarm_rate
            }, ignore_index=True)

            # Store event metrics
            self.event_metrics.append({
                'EventID': eventIDs[0] if eventIDs else 'N/A',
                'UserID': userIDs[0] if userIDs else 'N/A',
                'Sensitivity': sensitivity,
                'False_Positive_Rate': false_positive_rate,
                'False_Negative_Rate': false_negative_rate,
                'False_Alarm_Rate': overall_false_alarm_rate
            })

            # Update user-level metrics
            userID = userIDs[0] if userIDs else 'N/A'
            if userID != 'N/A':
                if userID not in self.user_metrics:
                    self.user_metrics[userID] = {'Sensitivity': 0, 'False_Positive_Rate': 0, 'False_Negative_Rate': 0, 'False_Alarm_Rate': 0, 'count': 0}

                self.user_metrics[userID]['Sensitivity'] += sensitivity
                self.user_metrics[userID]['False_Positive_Rate'] += false_positive_rate
                self.user_metrics[userID]['False_Negative_Rate'] += false_negative_rate
                self.user_metrics[userID]['False_Alarm_Rate'] += overall_false_alarm_rate
                self.user_metrics[userID]['count'] += 1

        # Save metrics summary
        self.metrics_summary.to_csv('/content/drive/MyDrive/event_metrics_summary.csv', index=False)

        # User summary
        user_summary_data = []
        for userID, metrics in self.user_metrics.items():
            if metrics['count'] > 0:
                user_summary_data.append({
                    'UserID': userID,
                    'Average_Sensitivity': metrics['Sensitivity'] / metrics['count'],
                    'Average_False_Positive_Rate': metrics['False_Positive_Rate'] / metrics['count'],
                    'Average_False_Negative_Rate': metrics['False_Negative_Rate'] / metrics['count'],
                    'Average_False_Alarm_Rate': metrics['False_Alarm_Rate'] / metrics['count'],
                })

        user_summary_df = pd.DataFrame(user_summary_data)
        user_summary_df.to_csv('user_event_metrics_summary.csv', index=False)

        # Save event metrics summary
        event_metrics_df = pd.DataFrame(self.event_metrics)
        event_metrics_df.to_csv('event_level_metrics_summary.csv', index=False)

        print("Metrics summary saved.")


In [41]:
class DataFormatter:
    def __init__(self, config):
        self.config = config

    def format_data(self, df_labels, test_split=0.3):
        """
        Split data into training and testing sets based on eventID, while ensuring proper row order is maintained.

        :param df_labels: DataFrame containing the dataset with 'eventId' column.
        :param test_split: Fraction of data to use for testing.
        :return: X_train_reshaped, X_test_reshaped, y_train, y_test
        """
        # Strip any unwanted spaces in column names (to ensure there are no hidden characters)
        df_labels.columns = df_labels.columns.str.strip()

        # Check if 'eventId' column is present
        if 'eventId' not in df_labels.columns:
            raise ValueError("The DataFrame must contain an 'eventId' column.")

        # Validate segment structure
        if not all(isinstance(seg, np.ndarray) and seg.shape == (125, 3) for seg in df_labels['segments']):
            raise ValueError("Each segment in the 'segments' column must be an array of shape (125, 3).")

        # Group by 'eventId' to maintain event-wise split
        unique_event_ids = df_labels['eventId'].unique()
        num_test_event_ids = int(len(unique_event_ids) * test_split)
        test_event_ids = sample(list(unique_event_ids), num_test_event_ids)
        test_event_ids_set = set(test_event_ids)

        # Split into training and testing sets
        df_test = df_labels[df_labels['eventId'].isin(test_event_ids_set)]
        df_train = df_labels[~df_labels['eventId'].isin(test_event_ids_set)]

        # Extract segments and labels for training and testing sets
        X_train = np.stack(df_train['segments'].values)
        y_train = np.stack(df_train['labels'].values)
        X_test = np.stack(df_test['segments'].values)
        y_test = np.stack(df_test['labels'].values)

        # Reshape segments into feature-wise dictionaries
        X_train_reshaped = self._reshape_segments(X_train)
        X_test_reshaped = self._reshape_segments(X_test)

        return X_train_reshaped, X_test_reshaped, y_train, y_test

    def _reshape_segments(self, segments):
        """
        Reshapes the segments array into the required format with feature dimension.
        :param segments: The array of segments to reshape.
        :return: Reshaped segments in a dictionary format with feature keys.
        """
        reshaped_segments = {}
        num_samples, num_time_steps, num_features = segments.shape
        for i in range(num_features):
            feature_name = f"Feature_{i+1}"
            reshaped_segments[feature_name] = segments[:, :, i].reshape(-1, num_time_steps, 1)
        return reshaped_segments


In [42]:
# Initialize DataLoader
data_loader = DataLoader(dataframe=processed_df, time_steps=Config.N_TIME_STEPS, step=Config.step, target_column='label')

# Load data (this will return a DataFrame with segments, labels, eventID, and userID)
df_labels = data_loader.load_data()
df_labels


C:\Users\jamie\AppData\Local\Temp\ipykernel_30636\1663183247.py:24: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  label_mode = stats.mode(group[self.target_column][i: i + self.time_steps])


,segments,labels,eventId,userID
0,"[[1066.9883, 16.4196, 1.1368684e-13], [1066.98...","[0.0, 1.0]",115,39
1,"[[1057.4006, 31.9105, 0.0], [1057.4006, 31.910...","[0.0, 1.0]",115,39
2,"[[1300.1477, 87.5605, 1.5916157e-12], [1300.14...","[0.0, 1.0]",115,39
3,"[[1653.6069, 91.1923, 5.684342e-13], [1653.606...","[0.0, 1.0]",115,39
4,"[[1506.6837, 102.189, 7.9580786e-13], [1506.68...","[0.0, 1.0]",115,39
...,...,...,...,...
1893,"[[1008.3253, -0.7043, 2.728484e-12], [1008.325...","[1.0, 0.0]",53666,39
1894,"[[1010.75024, -1.8391, 1.0231815e-12], [1010.7...","[1.0, 0.0]",53666,39
1895,"[[1007.9762, -1.0596, 1.4779289e-12], [1007.97...","[1.0, 0.0]",53666,39
1896,"[[985.01575, -1.0292, 5.684342e-13], [985.0157...","[1.0, 0.0]",53666,39


In [43]:
# Initialize DataFormatter
data_formatter = DataFormatter(config=Config)

# Split the data into train and test sets (30% test split)
X_train_reshaped, X_test_reshaped, y_train, y_test = data_formatter.format_data(df_labels, test_split=0.3)

# Check the output
print("Train set shape:", {key: val.shape for key, val in X_train_reshaped.items()})
print("Test set shape:", {key: val.shape for key, val in X_test_reshaped.items()})

# Combine features into a single array for model input
X_train_combined = np.concatenate([X_train_reshaped[key] for key in X_train_reshaped.keys()], axis=2)
X_test_combined = np.concatenate([X_test_reshaped[key] for key in X_test_reshaped.keys()], axis=2)

# Initialize model with residual fusion layer
ts_model = Amber_RF(row_hidden=Config.row_hidden, col_hidden=Config.row_hidden, num_classes=Config.N_CLASSES)

# Train the model
ts_model.fit(X_train_combined, y_train, validation_data=(X_test_combined, y_test), epochs=20, batch_size=32)

# Evaluate the model
test_loss, test_accuracy = ts_model.evaluate(X_test_combined, y_test)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")


Train set shape: {'Feature_1': (1302, 125, 1), 'Feature_2': (1302, 125, 1), 'Feature_3': (1302, 125, 1)}
Test set shape: {'Feature_1': (596, 125, 1), 'Feature_2': (596, 125, 1), 'Feature_3': (596, 125, 1)}


AttributeError: 'Amber_RF' object has no attribute 'fit'

In [109]:
df_labels

,segments,labels,eventId,userID
0,"[[1066.9883, 16.4196, 1.1368684e-13], [1066.98...","[0.0, 1.0]",115,39
1,"[[1057.4006, 31.9105, 0.0], [1057.4006, 31.910...","[0.0, 1.0]",115,39
2,"[[1300.1477, 87.5605, 1.5916157e-12], [1300.14...","[0.0, 1.0]",115,39
3,"[[1653.6069, 91.1923, 5.684342e-13], [1653.606...","[0.0, 1.0]",115,39
4,"[[1506.6837, 102.189, 7.9580786e-13], [1506.68...","[0.0, 1.0]",115,39
...,...,...,...,...
1893,"[[1008.3253, -0.7043, 2.728484e-12], [1008.325...","[1.0, 0.0]",53666,39
1894,"[[1010.75024, -1.8391, 1.0231815e-12], [1010.7...","[1.0, 0.0]",53666,39
1895,"[[1007.9762, -1.0596, 1.4779289e-12], [1007.97...","[1.0, 0.0]",53666,39
1896,"[[985.01575, -1.0292, 5.684342e-13], [985.0157...","[1.0, 0.0]",53666,39


In [102]:
processed_df.columns = processed_df.columns.str.strip()


In [108]:
# Initialize DataLoader
data_loader = DataLoader(dataframe=processed_df, time_steps=Config.N_TIME_STEPS, step=Config.step, target_column='label')

# Load data (this will return a DataFrame with segments, labels, eventID, and userID)
df_labels = data_loader.load_data()

# Initialize DataFormatter
data_formatter = DataFormatter(config=Config)

# Split the data into train and test sets (30% test split)
X_train_reshaped, X_test_reshaped, y_train, y_test = data_formatter.format_data(df_labels, test_split=0.3)

# Check the output
print("Train set shape:", {key: val.shape for key, val in X_train_reshaped.items()})
print("Test set shape:", {key: val.shape for key, val in X_test_reshaped.items()})

# Reshape y_test correctly
y_test_reshaped = np.asarray(y_test, dtype=np.float32)

# Initialize model with residual fusion layer
ts_model = Amber_RF(row_hidden=Config.row_hidden, col_hidden=Config.row_hidden, num_classes=Config.N_CLASSES)

# Create an instance of KFoldCrossValidation
kfold_cv = KFoldCrossValidation(ts_model, [X_train_reshaped['Feature_1'], X_train_reshaped['Feature_2'], X_test_reshaped['Feature_3']], y_train)

# Run the cross-validation
kfold_cv.run()

# Evaluate the model performance
evaluation_results = EventMetricsEvaluator(ts_model, [X_test_reshaped['Feature_1'], X_test_reshaped['Feature_2'], X_test_reshaped['Feature_3']], y_test_reshaped)


C:\Users\jamie\AppData\Local\Temp\ipykernel_7380\1663183247.py:24: FutureWarning: Unlike other reduction functions (e.g. `skew`, `kurtosis`), the default behavior of `mode` typically preserves the axis it acts along. In SciPy 1.11.0, this behavior will change: the default value of `keepdims` will become False, the `axis` over which the statistic is taken will be eliminated, and the value None will no longer be accepted. Set `keepdims` to True or False to avoid this warning.
  label_mode = stats.mode(group[self.target_column][i: i + self.time_steps])


ValueError: The DataFrame must contain 'eventId' and 'Id' columns.

NameError: name 'X_train' is not defined